In [ ]:
### Time frames - Define the start and end dates of the data that you want to look at (e.g. all 29 years between 1996 and 2025) (ICMECAT only goes to mid-2025 for now)
dataTimeframe = ["1996-01-01 00:00", "2025-01-01 00:00"]
start = dataTimeframe[0] # start date of data
end = dataTimeframe[1] # end date of data
# dataSourcesToUse = ["Wind"]
fileName = "Wind_" + str(start)[0:10] + "-" + str(end)[0:10]+ "_InputOutput.csv"

slowCutOff = 400
fastCutOff = 500

cmeBuffer = 12

# Get imports and functions needed
from sunpy.timeseries import GenericTimeSeries
import matplotlib.pyplot as plt
import numpy as np 
from datetime import timedelta
import os
import pandas as pd
import calendar
import warnings

import astropy.units as u
import astropy.constants as const
from astropy.visualization import quantity_support
warnings.filterwarnings("ignore")


/home/rahyndman/miniforge3/envs/sunpyVenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Wind

## Read in dataframe

In [3]:
windData = pd.read_csv(fileName, index_col=["Unnamed: 0"])

In [4]:
# print(windData.head(10))
print(windData.columns)

Index(['Year', 'DOY', 'Hour', 'Minute', 'IMF_PTS', 'PercentInterp',
       'CP/MVFlag', 'Timeshift', 'PFN_XGSE', 'PFN_YGSE', 'PFN_ZGSE', 'ScalarB',
       'Bx_GSEGSM', 'ByGSE', 'BzGSE', 'ByGSM', 'BzGSM', 'RMSTimeshift',
       'RMS_PFN', 'RMSScalarB', 'RMSFieldVector', 'Plasma_PTS', 'FlowSpeed',
       'Vx_GSE', 'Vy_GSE', 'Vz_GSE', 'ProtonDensity', 'Temp', 'X_s_c_GSE',
       'Y_s_c_GSE', 'Z_s_c_GSE', 'X_targ_GSE', 'Y_targ_GSE', 'Z_targ_GSE',
       'RMS_targ', 'DBOT1', 'DBOT2'],
      dtype='str')


## Add parameters

In [5]:
# B  = windData["ScalarB"].values * u.nT
Bx = windData["Bx_GSEGSM"].values* u.nT
By = windData["ByGSE"].values * u.nT
Bz = windData["BzGSE"].values* u.nT
# v  = windData["FlowSpeed"].values * u.km / u.s
Vx  = windData["Vx_GSE"].values * u.km / u.s
Vy  = windData["Vy_GSE"].values * u.km / u.s
Vz  = windData["Vz_GSE"].values * u.km / u.s
n  = windData["ProtonDensity"].values * u.cm**-3
T  = windData["Temp"].values * u.K

In [6]:
print(Vx[0:10])

[   nan    nan    nan -434.5 -434.5    nan -423.2 -423.8 -424.5 -424.5] km / s


In [7]:
# total solar wind speed
TotVsw = np.sqrt(Vx**2 + Vy**2 + Vz**2)

# magnitude of magnetic field
BMag = np.sqrt(Bx**2 + By**2 + Bz**2)

#Alfvén speed
vA = BMag / np.sqrt(const.mu0 * n * const.m_p)
vA = vA.to(u.km / u.s)

#Alfvén mach number
Ma = (TotVsw / vA).to(u.dimensionless_unscaled)

#Dynamic pressure
Pdyn = (n * const.m_p * TotVsw**2).to(u.nPa)
# Pdyn_by_hand = 1.6726e-6 * windData["ProtonDensity"] * windData["FlowSpeed"]**2 

#Beta
beta = (n * const.k_B * T / (BMag**2 / (2 * const.mu0))).to(u.dimensionless_unscaled)

#Cone and clock angles
cone = np.arccos(np.clip(Bx / BMag, -1, 1)).to(u.deg)    
clock = np.arctan2(By, Bz).to(u.deg)  



# spacecraft locations
R = 1 * u.au # for now set to 1 au, later use some conversion from -> windData["X_s_c_GSE"], windData["Y_s_c_GSE"], windData["Z_s_c_GSE"]
lat = 0 * u.deg # for now set to 0 degrees, later use some conversion from -> windData["X_s_c_GSE"], windData["Y_s_c_GSE"], windData["Z_s_c_GSE"]

In [8]:
windData["TotVsw"]    = TotVsw.to_value(u.km / u.s)
windData["BMag"]      = BMag.to_value(u.nT)
windData["vA_kms"]    = vA.to_value(u.km / u.s)
windData["Ma"]        = Ma.value
windData["Pdyn_nPa"]  = Pdyn.to_value(u.nPa)
windData["beta"]      = beta.value
windData["cone_deg"]  = cone.to_value(u.deg)
windData["clock_deg"] = clock.to_value(u.deg)

windData["r"]  = R.to_value(u.au)
windData["lat"] = lat.to_value(u.deg)

windData.describe().round(2)

,Year,DOY,Hour,Minute,IMF_PTS,PercentInterp,CP/MVFlag,Timeshift,PFN_XGSE,PFN_YGSE,...,TotVsw,BMag,vA_kms,Ma,Pdyn_nPa,beta,cone_deg,clock_deg,r,lat
count,15213886.00,15213886.00,15213886.00,15213886.00,15213886.00,15213886.00,15213886.00,15213886.00,15213886.00,15213886.00,...,11192612.00,13523203.00,11090241.00,11090241.00,11192612.00,11090241.00,13523203.00,13523203.00,15213886.0,15213886.0
mean,2010.02,183.18,11.50,29.50,115.56,157.74,5.10,113636.67,11.65,11.25,...,424.41,5.58,53.17,10.09,1.89,0.90,90.13,-0.87,1.0,0.0
std,8.37,105.58,6.92,17.32,312.40,300.51,4.58,313407.45,31.24,31.38,...,98.71,3.04,32.04,6.91,1.53,25.78,41.08,100.89,0.0,0.0
min,1996.00,1.00,0.00,0.00,1.00,0.00,0.00,-10797.00,0.01,-1.00,...,228.71,0.02,0.12,0.46,0.02,0.00,0.00,-179.98,1.0,0.0
25%,2003.00,92.00,5.00,14.00,3.00,0.00,1.00,2041.00,0.46,-0.07,...,350.57,3.67,34.27,6.81,1.09,0.27,56.22,-91.39,1.0,0.0
50%,2010.00,184.00,11.00,29.00,4.00,80.00,1.00,3200.00,0.64,0.28,...,401.85,4.91,47.57,8.92,1.53,0.52,90.09,-2.32,1.0,0.0
75%,2017.00,275.00,17.00,44.00,8.00,100.00,9.90,4461.00,0.84,0.59,...,477.03,6.66,64.13,11.61,2.22,0.92,124.05,89.76,1.0,0.0
max,2024.00,366.00,23.00,59.00,999.00,999.00,9.90,999999.00,99.99,99.99,...,1076.61,70.11,1121.47,3196.61,87.48,73034.12,180.00,180.00,1.0,0.0


In [9]:
# quantity_support()
# fig, axes = plt.subplots(5, 1, figsize=(9, 9), sharex=True)
# axes[0].plot(windData.index, BMag)
# axes[1].plot(windData.index, TotVsw)
# axes[2].plot(windData.index, Pdyn)
# axes[3].plot(windData.index, beta)
# axes[3].set_yscale("log")
# axes[4].plot(windData.index, Ma)

# for ax, name in zip(axes, ["B", "v", "Pdyn", "beta", "Ma"]):
#     ax.set_title(name, loc="left", fontsize=9)

# fig.tight_layout()

## Add CME column

In [10]:
## reading the catalogue
icmes = pd.read_csv("HELIO4CAST_ICMECAT_v23.csv")
icmes_wind = icmes[icmes["sc_insitu"]=="Wind"]
print(icmes_wind)

## just making these times more consistent to be like the df ones
## the .tz_localise to to remove the fact that the catalogue has the datatimes Z at the end.
for column in ["icme_start_time", "mo_start_time", "mo_end_time"]:
    icmes_wind[column] = pd.to_datetime(icmes_wind[column]).dt.tz_localize(None)

## here what we can do is create an ICME column with the same index as the dataframe. 
## then set it all to 0, then look at the time ranges from the catalogue and set 
## this column equal to sheath as 1, cme (magnetic object as 2), and then the rest as 0. you can also just change the start times and end times here etc. 
icme_column = pd.Series(np.int8(0), index=pd.to_datetime(windData.index))

print(icme_column)
for ICMEstart, ICMEend in zip(icmes_wind["icme_start_time"]-timedelta(hours=cmeBuffer), icmes_wind["mo_start_time"]):
    icme_column.loc[ICMEstart:ICMEend] = 1
for ICMEstart, ICMEend in zip(icmes_wind["mo_start_time"], icmes_wind["mo_end_time"]+timedelta(hours=cmeBuffer)):
    icme_column.loc[ICMEstart:ICMEend] = 2

print(icme_column)

windData["cmes"] = icme_column
print(windData["cmes"])

#Laura's note
# you can also change the icme_start_time to be icme_start_time - 12 hours etc. this is just directly from the times from the catalogue.
# then you can do nice things like:
# no_cmes = df.where(df["cmes"] == 0)
# or no_cmes = df[df["cmes"]==0] etc.

      Unnamed: 0.1  Unnamed: 0                    icmecat_id sc_insitu  \
1                1           1  ICME_Wind_MOESTL_20250726_01      Wind   
4                4           4  ICME_Wind_MOESTL_20250703_01      Wind   
9                9           9  ICME_Wind_MOESTL_20250612_01      Wind   
10              10          10  ICME_Wind_MOESTL_20250607_01      Wind   
11              11          11  ICME_Wind_MOESTL_20250604_01      Wind   
...            ...         ...                           ...       ...   
1908          1908        1908    ICME_Wind_NASA_19950405_01      Wind   
1909          1909        1909    ICME_Wind_NASA_19950403_01      Wind   
1910          1910        1910    ICME_Wind_NASA_19950306_01      Wind   
1911          1911        1911    ICME_Wind_NASA_19950304_01      Wind   
1912          1912        1912    ICME_Wind_NASA_19950207_01      Wind   

        icme_start_time      mo_start_time        mo_end_time  \
1     2025-07-26T16:44Z  2025-07-26T16:44Z  20

In [11]:
# icmes_wind["icme_start_time"].iloc[532:540], icmes_wind["mo_start_time"][532:540]
# windData[0:int(365.25*24*60 + 1080)]
# windData[0:int(365.25*24*60 + 1080)]["cmes"].unique()
# icme_column[0:int(365.25*24*60 + 1080)]
# icme_column[0:int(365.25*24*60 + 1080)].unique()

## Add sw type column

In [12]:
def wind_type(speed):
    if np.isnan(speed):
        return "no data"
    elif speed > fastCutOff:
        return "fast"
    elif speed > slowCutOff:
        return "intermediate"
    else:
        return "slow"

windData["sw_type"] = windData["TotVsw"].apply(wind_type)

## Drop not-needed columns

In [15]:
windData.columns

Index(['Year', 'DOY', 'Hour', 'Minute', 'IMF_PTS', 'PercentInterp',
       'CP/MVFlag', 'Timeshift', 'PFN_XGSE', 'PFN_YGSE', 'PFN_ZGSE', 'ScalarB',
       'Bx_GSEGSM', 'ByGSE', 'BzGSE', 'ByGSM', 'BzGSM', 'RMSTimeshift',
       'RMS_PFN', 'RMSScalarB', 'RMSFieldVector', 'Plasma_PTS', 'FlowSpeed',
       'Vx_GSE', 'Vy_GSE', 'Vz_GSE', 'ProtonDensity', 'Temp', 'X_s_c_GSE',
       'Y_s_c_GSE', 'Z_s_c_GSE', 'X_targ_GSE', 'Y_targ_GSE', 'Z_targ_GSE',
       'RMS_targ', 'DBOT1', 'DBOT2', 'TotVsw', 'BMag', 'vA_kms', 'Ma',
       'Pdyn_nPa', 'beta', 'cone_deg', 'clock_deg', 'r', 'lat', 'cmes',
       'sw_type'],
      dtype='str')

In [18]:
windDataDropped = windData.drop(columns=['Year', 'DOY', 'Hour', 'Minute', 'ScalarB', 'FlowSpeed'])

In [19]:
windDataDropped.columns

Index(['IMF_PTS', 'PercentInterp', 'CP/MVFlag', 'Timeshift', 'PFN_XGSE',
       'PFN_YGSE', 'PFN_ZGSE', 'Bx_GSEGSM', 'ByGSE', 'BzGSE', 'ByGSM', 'BzGSM',
       'RMSTimeshift', 'RMS_PFN', 'RMSScalarB', 'RMSFieldVector', 'Plasma_PTS',
       'Vx_GSE', 'Vy_GSE', 'Vz_GSE', 'ProtonDensity', 'Temp', 'X_s_c_GSE',
       'Y_s_c_GSE', 'Z_s_c_GSE', 'X_targ_GSE', 'Y_targ_GSE', 'Z_targ_GSE',
       'RMS_targ', 'DBOT1', 'DBOT2', 'TotVsw', 'BMag', 'vA_kms', 'Ma',
       'Pdyn_nPa', 'beta', 'cone_deg', 'clock_deg', 'r', 'lat', 'cmes',
       'sw_type'],
      dtype='str')

## Save to csv

In [20]:
windDataDropped.to_csv("Wind_" + str(start)[0:10] + "-" + str(end)[0:10]+ "_Enriched.csv")